# 🎛️ Ableton Session Intelligence Engine (H.O.R.N. Stack)

A forensic-grade, local-first metadata tracking and session intelligence layer for Ableton Live. This engine automatically intercepts compressed `.als` project XML graphs, flushes them into zero-copy Apache Arrow buffers, caches them inside a structured Parquet/DuckDB lakehouse, and uses local hardware-muzzled LLMs to reason over acoustic DSP states.

---

## ⚡ The H.O.R.N. Architectural Manifesto

*   **H — Hardware-Optimized:** Tensors, arrays, and model weights are pinned directly to local GPU VRAM/RAM pools with absolute zero cloud round-trip latencies.
*   **O — On-Demand / Offline:** 100% data privacy. No unreleased client audio or proprietary arrangement metadata is ever leaked to third-party web servers.
*   **R — Reproducible:** Structural ledger states are cataloged deterministically via DuckDB, LanceDB, and Parquet snapshots.
*   **N — Native:** The extraction layer hooks directly into native Ableton Live XML file formats without binary memory injections.

---

## 🧠 Production Engine Capabilities (`main.py`)

While this notebook acts as the core proof-of-concept scratchpad, the full production web service (`main.py`) elevates this stack into a fully autonomous Mix Assistant API:

*   **Acoustic DSP Cross-Referencing:** Utilizes SQL `JOIN` logic inside DuckDB to marry raw Ableton plugin data with an actual DSP acoustic database (`sonic_core_v2.duckdb`). Evaluates tracks across acoustic dimensions like **Tempo, Key, RMS Loudness, Crest Factor, and Sub-Bass Energy**.
*   **Two-Stage Pydantic Pipeline:** 
    1. Pre-validates the rigid DuckDB arrays via the `AlignedDSPTrackRecord` schema to prevent runtime poisoning.
    2. Enforces structural LLM outputs via the `AutomatedDAWDiagnostic` schema, ensuring deterministic JSON responses.
*   **Actionable Producer Feedback:** Generates hard technical verdicts (e.g., `CRITICAL`, `WARNING`) alongside a step-by-step `assistant_remediation_monologue` dictating exact EQ or compression moves required to un-brick a mix.
*   **API Hardware Bypass Switch:** Features a highly optimized `return_raw_only` endpoint flag. Allows local applications to bypass the heavy Ollama LLM execution entirely and instantly fetch the raw Parquet metadata arrays at sub-millisecond speeds.

---

## 📂 Repository File Architecture

*   `session_intel_pipeline_v1.ipynb`: This unified workspace notebook containing the core execution layer and environment instantiation.
*   `main.py`: Production entry point for launching the local ASGI web service (`uvicorn`) and serving the Pydantic-muzzled Phi-3 engine.
*   `pyproject.toml` / `requirements.txt`: Explicit dependency specifications and lock parameters.
*   `lakehouse_data/`: Local storage directory holding high-speed `.parquet` file extracts.
*   `raw_sessions/`: Input directory for compressed Ableton Live `.als` xml graphs.

---

## 🛠️ Instant Zero-Configuration Installation

This project requires **Python 3.13+** and a running local instance of **Ollama** (with the `phi3:3.8b` model pulled).

### 1. Isolate and Instantiate Your Virtual Environment
```bash
python -m venv .venv_fresh


In [1]:
# =====================================================================
# CELL 1: WORKSPACE SCRATCHPAD & FILE SYSTEM INITIALIZATION
# =====================================================================
import os
from pathlib import Path

# Ensure project directories exist
Path("./raw_sessions").mkdir(parents=True, exist_ok=True)
Path("./lakehouse_data").mkdir(parents=True, exist_ok=True)
Path("./app/templates").mkdir(parents=True, exist_ok=True)

# 1. Write pyproject.toml
pyproject_content = """[project]
name = "ableton-session-intelligence"
version = "1.0.0"
description = "Forensic-grade local session intelligence layer for Ableton Live"
readme = "README.md"
requires-python = ">=3.13"
dependencies = [
    "pydantic>=2.10",
    "pydantic-settings>=2.4",
    "duckdb>=1.1",
    "lancedb>=0.17",
    "pyarrow>=18.0",
    "jinja2>=3.1",
    "fastapi>=0.115",
    "uvicorn>=0.32",
    "ollama>=0.2"
]

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"
"""
with open("pyproject.toml", "w", encoding="utf-8") as f:
    f.write(pyproject_content)

# 2. Write requirements.txt
requirements_content = """duckdb>=1.1
lancedb>=0.17
pyarrow>=18.0
pydantic>=2.10
pydantic-settings>=2.4
jinja2>=3.1
fastapi>=0.115
uvicorn>=0.32
ollama>=0.2
torch
"""
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)

# 3. Write .env
env_content = """# --- CORE ENGINE SETTINGS ---
SESSION_INTEL_ENVIRONMENT_STAGE=production
SESSION_INTEL_OMP_NUM_THREADS=4
SESSION_INTEL_MODEL_CHOICE=phi3

# --- AUTOMATIC PATH ALLOCATIONS ---
SESSION_INTEL_INPUT_ALS_DIRECTORY=./raw_sessions
SESSION_INTEL_PARQUET_LAKEHOUSE_DIR=./lakehouse_data
SESSION_INTEL_LANCEDB_URI=./.vector_cache
"""
with open(".env", "w", encoding="utf-8") as f:
    f.write(env_content)

# 4. Write README.md
readme_content = """# 🎛️ Ableton Session Intelligence Engine (H.O.R.N. Stack)

A forensic-grade, local-first metadata tracking and session intelligence layer for Ableton Live. This engine automatically intercepts compressed `.als` project XML graphs, flushes them into zero-copy Apache Arrow buffers, caches them inside a structured Parquet/DuckDB lakehouse, and uses local hardware-muzzled LLMs to reason over session state histories.

---

## ⚡ The H.O.R.N. Architectural Manifesto

*   **H — Hardware-Optimized:** Tensors, arrays, and model weights are pinned directly to local GPU VRAM/RAM pools with absolute zero cloud round-trip latencies.
*   **O — On-Demand / Offline:** 100% data privacy. No unreleased client audio or proprietary arrangement metadata is ever leaked to third-party web servers.
*   **R — Reproducible:** Structural ledger states are cataloged deterministically via DuckDB, LanceDB, and Parquet snapshots.
*   **N — Native:** The extraction layer hooks directly into native Ableton Live XML file formats without binary memory injections.

---

## 📂 Repository File Architecture

*   `main.py`: Entry point for launching the local ASGI web service.
*   `pyproject.toml`: Explicit dependency specifications and lock parameters for Python 3.13.
*   `app/config.py`: Environment variable and file storage route controller managed via `pydantic-settings`.
*   `app/extractor.py`: High-performance Gzip-to-Arrow binary unpacker layer.
*   `app/database.py`: Embedded DuckDB OLAP file querying engine.
*   `app/schemas.py`: Pydantic V2 data model definitions managing your 3 explicit Data Lanes.
*   `app/api.py`: FastAPI server exposing the network endpoints to your local IDE or language servers.
*   `app/templates/`: Isolated directory mapping prompt files away from core logical code frames.

---

## 🛠️ Instant Zero-Configuration Installation

This project requires **Python 3.13+** and a running local instance of **Ollama** (with the `phi3` model pulled).

### 1. Clone the Asset Repository
```bash
git clone https://github.com
cd ableton-session-intelligence
```

### 2. Isolate and Instantiate Your Virtual Environment
```bash
python -m venv .venv
```
*   **Windows:** `.venv\\Scripts\\activate`
*   **Mac/Linux:** `source .venv/bin/activate`

### 3. Inject Strict Dependency Classes
```bash
pip install --upgrade pip
pip install duckdb lancedb pyarrow pydantic pydantic-settings jinja2 fastapi uvicorn ollama torch
```

---

## 🚀 Execution Guide

1.  Place your target Ableton Live project files (`.als`) into the `raw_sessions/` directory.
2.  Boot up your local terminal and execute the primary runtime engine node:
    ```bash
    python main.py
    ```
3.  The engine will automatically scan the project files, index track attributes to local matching Parquet records, and spin up a local FastAPI server hosted at `http://127.0.0.1:8000`.

---

## 🛡️ Commercial Licensing & Security Compliance
This engine processes user file streams completely locally. It reads user-generated XML data configurations and does not reverse-engineer, alter, or disassemble closed-source Ableton Live application binaries, making it fully compliant with standard software interoperability regulations.
"""
with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("✅ Project directories and environment config files (.env, pyproject.toml, requirements.txt, README.md) initialized.")

✅ Project directories and environment config files (.env, pyproject.toml, requirements.txt, README.md) initialized.


In [2]:
# =====================================================================
# CELL 2: REQUIREMENTS, EXPLICIT IMPORTS, & ENVIRONMENT VERIFICATION
# =====================================================================
import os
import sys
import gzip
import time
import json
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Literal, List, Dict, Any, Optional

# Core Local Data Lakehouse & Matrix Math Arrays
import torch
import numpy as np
import duckdb
import lancedb
import pyarrow as pa
import pyarrow.parquet as pq

# Local Language Model Connectivity & Microservices
import ollama
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from pydantic_settings import BaseSettings, SettingsConfigDict
from jinja2 import Template

print(f"\ud83d\udc0d Python Target Build: {sys.version}")
print(f"\ud83d\udd25 Hardware Acceleration - PyTorch GPU Active: {torch.cuda.is_available()}")
print(f"\u2705 Local Database Engine Modules Initialized and Bound.")

Exception in callback BaseAsyncIOLoop._handle_events()
handle: <Handle BaseAsyncIOLoop._handle_events()>
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 0-1: surrogates not allowed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\STUDIES_BACKUP\.venv_fresh\Lib\site-packages\jupyter_client\session.py", line 143, in orjson_packer
    return orjson.dumps(obj, default=json_default, option=option)
           ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: str is not valid UTF-8: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\STUDIES_BACKUP\.venv_fresh\Lib\site-packages\jupyter_client\session.py", line 103, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 28-29: surro

In [3]:
# =====================================================================
# CELL 3: BASE CONFIGURATION & THE 3 VALIDATED DATA LANES
# =====================================================================

class ProductionEngineSettings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="SESSION_INTEL_", env_file=".env", extra="ignore")

    # Path Resolution Boundaries
    project_root: Path = Path(os.getcwd())
    parquet_lakehouse_dir: Path = Path(os.getcwd()) / "lakehouse_data"
    lancedb_uri: Path = Path(os.getcwd()) / ".vector_cache"
    
    # Engine Settings
    environment_stage: Literal["development", "production"] = "production"
    omp_num_threads: int = 4
    model_choice: str = "phi3"

    def __init__(self, **values):
        super().__init__(**values)
        self.parquet_lakehouse_dir.mkdir(parents=True, exist_ok=True)

settings = ProductionEngineSettings()

# --- THE THREE EXPLICIT SYSTEM DATA LANES ---
class Lane1DuckDBAnalytics(BaseModel):
    """Lane 1: High-speed structural metric reporting and tabular abnormalities"""
    table_source: str = Field(..., description="Target Parquet data partition evaluated")
    processed_row_count: int
    anomalous_metric_detected: bool
    structural_monologue: str

class Lane2LanceDBVectors(BaseModel):
    """Lane 2: Semantic audio similarity tracking, vector keys, and file locations"""
    matched_vector_keys: List[int]
    maximum_semantic_distance: float
    nearest_file_nodes: List[str]

class Lane3SystemLogs(BaseModel):
    """Lane 3: Deep system tracking and core workspace layout status updates"""
    process_id: int
    execution_severity: Literal["INFO", "WARNING", "CRITICAL"]
    root_cause_analysis: str

print(f"\ud83d\udcbe Storage Path Initialized: {settings.parquet_lakehouse_dir.resolve()}")
print("\ud83d\udee1\ufe0f The 3 Pydantic Data Lanes have been registered into the active memory array.")

Exception in callback BaseAsyncIOLoop._handle_events()
handle: <Handle BaseAsyncIOLoop._handle_events()>
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 0-1: surrogates not allowed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\STUDIES_BACKUP\.venv_fresh\Lib\site-packages\jupyter_client\session.py", line 143, in orjson_packer
    return orjson.dumps(obj, default=json_default, option=option)
           ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: str is not valid UTF-8: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\STUDIES_BACKUP\.venv_fresh\Lib\site-packages\jupyter_client\session.py", line 103, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 28-29: surro

In [4]:
# =====================================================================
# CELL 4: MEMORY-MAPPED GZIP EXTRACTOR TO APACHE ARROW / PARQUET
# =====================================================================

def extract_and_index_session(input_path: str) -> str:
    """
    Unpacks Gzipped Ableton XML project file entirely in-memory and 
    streams track configurations straight to a matching local Parquet ledger.
    """
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"\u274c Input Ableton file missing at: {input_path}")
        
    # Standardize matching name convention for the output parquet target
    base_name = os.path.splitext(os.path.basename(input_path))[0]
    output_parquet_path = str(settings.parquet_lakehouse_dir / f"{base_name}.parquet")
    
    # Decompress Gzip data straight to RAM
    with gzip.open(input_path, 'rb') as f:
        xml_content = f.read()
        
    root = ET.fromstring(xml_content)
    
    track_ids, track_names, track_types, plugin_lists = [], [], [], []
    tracks_node = root.find(".//Tracks")
    
    if tracks_node is None:
        raise ValueError("Malformed file configuration: No <Tracks> collection located.")
        
    for track in tracks_node:
        t_id = int(track.get("Id", -1))
        name_node = track.find(".//Name/EffectiveName")
        t_name = name_node.get("Value", "Untitled") if name_node is not None else "Untitled"
        t_type = track.tag
        
        devices = []
        device_chain = track.find(".//DeviceChain/DeviceChain/Devices")
        if device_chain is not None:
            for device in device_chain:
                dev_name_node = device.find(".//UserName")
                dev_name = dev_name_node.get("Value", "") if dev_name_node is not None else ""
                if not dev_name:
                    dev_name = device.tag
                devices.append(dev_name)
                
        track_ids.append(t_id)
        track_names.append(t_name)
        track_types.append(t_type)
        plugin_lists.append(devices)

    # Instantiate zero-copy Apache Arrow Data Representation Table
    schema = pa.schema([
        ('track_id', pa.int64()),
        ('track_name', pa.string()),
        ('track_type', pa.string()),
        ('active_plugins', pa.list_(pa.string()))
    ])
    arrow_table = pa.Table.from_arrays(
        [track_ids, track_names, track_types, plugin_lists], schema=schema
    )
    
    # Commit binary array directly to local data storage
    pq.write_table(arrow_table, output_parquet_path)
    return output_parquet_path

print("\u26a1 Gzip Extractor and PyArrow Conversion layers ready for execution.")

⚡ Gzip Extractor and PyArrow Conversion layers ready for execution.


In [5]:
# =====================================================================
# CELL 5: ISOLATED JINJA PROMPT TEMPLATE BLUEPRINT
# =====================================================================

JINJA_SESSION_EXAMINER = """
System: You are an expert Ableton Live structural engine evaluator.
Analyze the following tracks extracted from a local user session file via DuckDB.
You must output a single JSON object matching the requested schema layout.

Session Track Dump:
{% for track in track_data %}
* Track #{{ track[0] }}: "{{ track[1] }}" (Type: {{ track[2] }}) -> Active Processing: {{ track[3] }}
{% endfor %}

User Query: {{ query }}
"""

# Save template to local app directory for modular structure
with open("./app/templates/session_prompt.jinja", "w", encoding="utf-8") as f:
    f.write(JINJA_SESSION_EXAMINER.strip())

print("\ud83c\udf9b\ufe0f Master Jinja Prompt Template text compiled, saved to app/templates/session_prompt.jinja, and locked in memory.")

Exception in callback BaseAsyncIOLoop._handle_events()
handle: <Handle BaseAsyncIOLoop._handle_events()>
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 0-1: surrogates not allowed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\STUDIES_BACKUP\.venv_fresh\Lib\site-packages\jupyter_client\session.py", line 143, in orjson_packer
    return orjson.dumps(obj, default=json_default, option=option)
           ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: str is not valid UTF-8: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\STUDIES_BACKUP\.venv_fresh\Lib\site-packages\jupyter_client\session.py", line 103, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 28-29: surro

In [6]:
# =====================================================================
# CELL 6: DETACHED REASONING CONNECTIVITY LAYER
# =====================================================================

def analyze_session_with_phi3(parquet_file_path: str, user_request: str) -> Lane1DuckDBAnalytics:
    """
    Queries cold storage Parquet rows via projection pushdowns and uses 
    local Phi-3 constraints to return verified, token-muzzled Pydantic analytics.
    """
    con = duckdb.connect(database=":memory:")
    con.execute(f"SET threads={settings.omp_num_threads};")
    
    # Extract records directly out of your local binary Parquet files
    raw_db_rows = con.execute(f"""
        SELECT track_id, track_name, track_type, active_plugins 
        FROM read_parquet('{parquet_file_path}')
        ORDER BY track_id ASC
    """).fetchall()
    
    # Compile prompt by feeding rows into separate Jinja template
    jinja_compiler = Template(JINJA_SESSION_EXAMINER)
    fully_rendered_prompt = jinja_compiler.render(
        track_data=raw_db_rows,
        query=user_request
    )
    
    # Pass the prompt to your offline model execution engine
    response = ollama.chat(
        model=settings.model_choice,
        messages=[{"role": "user", "content": fully_rendered_prompt}],
        options={"temperature": 0.0},
        format=Lane1DuckDBAnalytics.model_json_schema()  # <-- THE MUZZLE LINK
    )
    
    raw_ai_string = response['message']['content']
    validated_insight = Lane1DuckDBAnalytics.model_validate_json(raw_ai_string)
    
    return validated_insight

print("\ud83e\udde0 Offline Muzzled Inference Routing Engine fully bound and operational.")

Exception in callback BaseAsyncIOLoop._handle_events()
handle: <Handle BaseAsyncIOLoop._handle_events()>
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 0-1: surrogates not allowed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\STUDIES_BACKUP\.venv_fresh\Lib\site-packages\jupyter_client\session.py", line 143, in orjson_packer
    return orjson.dumps(obj, default=json_default, option=option)
           ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: str is not valid UTF-8: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\STUDIES_BACKUP\.venv_fresh\Lib\site-packages\jupyter_client\session.py", line 103, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 28-29: surro

In [7]:
# =====================================================================
# CELL 7: FastAPI PRODUCTION REST ROUTER MOUNT
# =====================================================================
app = FastAPI(
    title="Ableton Session Intelligence Node",
    version="1.0.0",
    description="H.O.R.N. Stack Local-First Execution Node."
)

class QueryPayload(BaseModel):
    session_file_path: str
    prompt_query: str

@app.post("/v1/agent/inference")
async def execute_pipeline_endpoint(payload: QueryPayload):
    try:
        # Step 1: Unpack and create matching Parquet files
        parquet_target = extract_and_index_session(payload.session_file_path)
        
        # Step 2: Route through DuckDB and your muzzled local Phi-3 engine
        validated_data = analyze_session_with_phi3(parquet_target, payload.prompt_query)
        
        return {
            "status": "success",
            "engine": "Ollama-Phi3",
            "lane_executed": 1,
            "data": validated_data.model_dump()
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("🌍 FastAPI Router mounted successfully. Production API live on http://127.0.0.1:8000")


🌍 FastAPI Router mounted successfully. Production API live on http://127.0.0.1:8000


In [8]:
# =====================================================================
# CELL 8: PIPELINE CALL TEMPLATE
# =====================================================================
payload = {
    "session_file_path": "./raw_sessions/2-16 BIG DRIP TALK.als",
    "prompt_query": "Identify all MidiTracks running third-party virtual instruments."
}

print("Ready. Place an Ableton file named YourProjectName.als in ./raw_sessions and uncomment the line below to test:")
await execute_pipeline_endpoint(QueryPayload(**payload))

Ready. Place an Ableton file named YourProjectName.als in ./raw_sessions and uncomment the line below to test:


{'status': 'success',
 'engine': 'Ollama-Phi3',
 'lane_executed': 1,
 'data': {'table_source': 'session_tracks',
  'processed_row_count': 10,
  'anomalous_metric_detected': False,
  'structural_monologue': "[{'track_id': '386', 'title': 'GHOST kick', 'type': 'MidiTrack', 'active_processing': ['Eq8']}, {'track_id': '413', 'title': 'drum buss', 'type': 'GroupTrack', 'active_processing': ['AudioEffectGroupDevice']}]"}}